<a href="https://colab.research.google.com/github/Karandeep09/transformer/blob/main/MaskedSelfAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import torch
import torch.nn as nn
import math

class MaskedSelfAttention(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.embed_size = embed_size
    self.q_w = nn.Linear(in_features=embed_size, out_features=embed_size, bias=False)
    self.k_w = nn.Linear(in_features=embed_size, out_features=embed_size, bias=False)
    self.v_w = nn.Linear(in_features=embed_size, out_features=embed_size, bias=False)
  def forward(self, embeddings, mask=None):
    q = self.q_w(embeddings)
    k = self.k_w(embeddings)
    v = self.v_w(embeddings)
    sims = q @ k.transpose(-1, -2)
    scaled_sims = sims / math.sqrt(k.size(-1))
    if mask is not None:
      scaled_sims = scaled_sims.masked_fill(mask = mask, value = -1e9)
    attention_percents = torch.softmax(scaled_sims, dim=-1)
    print("Attention Percents = ", attention_percents)
    return attention_percents @ v


selfAttention = MaskedSelfAttention(2)
token_embeddings = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
tokens = token_embeddings.size(-2)
mask = torch.tril(torch.ones(tokens, tokens))
mask = mask == 0
print("Mask = ", mask)
attention_scores = selfAttention(token_embeddings, mask)
print("Masked Self Attention Scores = ", attention_scores)

Mask =  tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
Attention Percents =  tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00],
        [9.6550e-02, 9.0345e-01, 0.0000e+00],
        [8.8978e-04, 2.9374e-02, 9.6974e-01]], grad_fn=<SoftmaxBackward0>)
Masked Self Attention Scores =  tensor([[-0.8990, -0.9304],
        [-1.2934, -2.2081],
        [-1.7584, -3.7149]], grad_fn=<MmBackward0>)


In [13]:
import math

# 1. Print the transposed weight matrices of the linear layers
print("q_w transpose:", selfAttention.q_w.weight.transpose(-2, -1))
print("k_w transpose:", selfAttention.k_w.weight.transpose(-2, -1))
print("v_w transpose:", selfAttention.v_w.weight.transpose(-2, -1))

# 2. Perform manual operations
q = token_embeddings @ selfAttention.q_w.weight.transpose(-2, -1)
k = token_embeddings @ selfAttention.k_w.weight.transpose(-2, -1)
v = token_embeddings @ selfAttention.v_w.weight.transpose(-2, -1)

sims = q @ k.transpose(-2, -1)
scaled_sims = sims / math.sqrt(k.size(-1))

# Apply the same mask logic
manual_scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)
manual_attention_percents = torch.softmax(manual_scaled_sims, dim=-1)
manual_output = manual_attention_percents @ v

# 3. Compare with the module output
print("Attention Scores:", attention_scores)
print("Manual Output:", manual_output)
print("Matches:", torch.allclose(attention_scores, manual_output))

q_w transpose: tensor([[-0.4492,  0.3106],
        [-0.3541,  0.3252]], grad_fn=<TransposeBackward0>)
k_w transpose: tensor([[ 0.4742,  0.5145],
        [-0.1923,  0.5428]], grad_fn=<TransposeBackward0>)
v_w transpose: tensor([[ 0.4625, -0.4839],
        [-0.6808, -0.2233]], grad_fn=<TransposeBackward0>)
Attention Scores: tensor([[-0.8990, -0.9304],
        [-1.2934, -2.2081],
        [-1.7584, -3.7149]], grad_fn=<MmBackward0>)
Manual Output: tensor([[-0.8990, -0.9304],
        [-1.2934, -2.2081],
        [-1.7584, -3.7149]], grad_fn=<MmBackward0>)
Matches: True
